## 1. Project Overview

This project develops a comprehensive Python-based "Institutional Factor Investing Engine." It aims to demonstrate the end-to-end process of building a quantitative investment strategy, from data acquisition and cleaning to factor estimation, portfolio optimization, backtesting, and performance attribution. The engine will focus on implementing Fama-French factor models to identify and exploit systematic risk premiums in equity markets, suitable for a professional quantitative developer or Python engineer's portfolio.

### 2. Real-World Finance Use Case

**Enhancing Equity Portfolio Management with Factor Insights**

In institutional finance, portfolio managers and quantitative analysts continuously seek strategies to generate alpha (returns in excess of a benchmark) and better understand the drivers of portfolio risk and return. This factor investing engine directly addresses this need by providing a systematic approach to:

1.  **Strategic Asset Allocation:** By understanding and targeting specific factors (e.g., Value, Size, Momentum, Quality), institutions can construct portfolios with desired risk-return characteristics, aligning with their investment mandates and risk tolerance.
2.  **Risk Management:** Factor exposures allow for a deeper understanding of portfolio risk beyond traditional asset-level diversification. Managers can monitor and manage unwanted factor bets or explicitly target desired factor risks.
3.  **Performance Attribution:** The engine will help explain why a portfolio performed the way it did, attributing returns to market exposure, specific factor exposures (e.g., Fama-French factors), and idiosyncratic alpha. This is crucial for reporting to clients and internal strategy refinement.
4.  **Portfolio Construction & Optimization:** It provides a framework for optimizing portfolios based on factor exposures, aiming to maximize risk-adjusted returns (e.g., Sharpe Ratio) while adhering to practical constraints (e.g., long-only positions, diversification).
5.  **Quantitative Strategy Development:** Serves as a foundation for developing and backtesting more advanced quantitative strategies, incorporating new factors, alternative risk models, or machine learning techniques to predict factor premiums.

## Environment Setup and Configuration

In [ ]:
# Install necessary libraries
!pip install pandas numpy yfinance statsmodels PyPortfolioOpt plotly fredapi

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
from datetime import datetime
import requests
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# --- Configuration Parameters ---
# These parameters would typically be in a config.py file for a larger project

START_DATE = '2010-01-01'
END_DATE = '2023-12-31'

# Example tickers for a broad market portfolio
# For a true institutional project, this would be a dynamic list (e.g., S&P 500 constituents)
# For demonstration, we'll pick a few well-known large-cap stocks.
# Note: yfinance can sometimes fail for a few tickers, or have partial data.
# In a production system, robust error handling and alternative data sources would be used.
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'JPM', 'V', 'PG', 'XOM', 'NFLX']

MARKET_TICKER = '^GSPC' # S&P 500 for market benchmark

# FRED API Key (replace with your actual key or load from environment variables/Colab secrets)
# FRED_API_KEY = 'YOUR_FRED_API_KEY'
# For this notebook, we might try to proceed without a key if the data is publicly accessible
# or if we are using a pre-downloaded dataset for demonstration.
# For robustness, using a key is recommended. If you have one, uncomment and replace.

print(f"Project configured for data from {START_DATE} to {END_DATE}")
print(f"Selected tickers: {', '.join(TICKERS)}")


## 8. Data Collection Pipeline Implementation

### 8.1 Stock Prices (Yahoo Finance via `yfinance`)

This section downloads historical daily adjusted closing prices for the specified stock tickers and the market benchmark from Yahoo Finance. The data is stored in a DataFrame and potential issues like missing data for specific tickers are noted.

In [ ]:
def download_stock_data(tickers, start_date, end_date):
    """
    Downloads historical adjusted closing prices for a list of tickers.

    Args:
        tickers (list): List of stock ticker symbols.
        start_date (str): Start date for data download (YYYY-MM-DD).
        end_date (str): End date for data download (YYYY-MM-DD).

    Returns:
        pd.DataFrame: DataFrame with adjusted closing prices, indexed by date.
    """
    print(f"Downloading stock data for {len(tickers)} tickers from {start_date} to {end_date}...")
    data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']

    # Ensure data is a DataFrame, even for a single ticker download
    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])

    # Handle cases where yfinance might return empty or partially empty data for some tickers
    if data.empty:
        print("Warning: No stock data downloaded. Check tickers and dates.")
        return pd.DataFrame()

    print("Stock data download complete.")
    return data

# Download individual stock prices
stock_prices = download_stock_data(TICKERS, START_DATE, END_DATE)

# Download market benchmark prices
market_prices = download_stock_data([MARKET_TICKER], START_DATE, END_DATE)

# Rename market column for clarity
if not market_prices.empty:
    market_prices = market_prices.rename(columns={MARKET_TICKER: 'Market'})

print("\n--- Stock Prices (first 5 rows) ---")
display(stock_prices.head())
print("\n--- Market Prices (first 5 rows) ---")
display(market_prices.head())


### 8.2 Fama-French Factor Data (Kenneth French Data Library)

This section retrieves the Fama-French 5-Factor (or 3-Factor) daily data from Kenneth French's data library. We'll download the CSV directly and process it into a pandas DataFrame. This includes cleaning column names and converting to appropriate data types.

In [ ]:
def download_fama_french_factors(url, start_date, end_date):
    """
    Downloads Fama-French factor data from a specified URL (Kenneth French's library).

    Args:
        url (str): URL to the Fama-French factor data CSV/TXT file.
        start_date (str): Start date for data (YYYY-MM-DD).
        end_date (str): End date for data (YYYY-MM-DD).

    Returns:
        pd.DataFrame: DataFrame with Fama-French factors, indexed by date.
    """
    print(f"Downloading Fama-French factor data from {url}...")
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        # Kenneth French data usually has a header with meta-info before the actual data
        # We need to find where the actual data starts
        data_lines = response.text.splitlines()
        start_line = 0
        for i, line in enumerate(data_lines):
            if line.strip().startswith('Mkt-RF') or line.strip().startswith('Mkt_Rf'): # Adjust based on actual header
                start_line = i
                break

        data_io = StringIO('\n'.join(data_lines[start_line:]))

        # Read the data, skipping the initial descriptive lines
        # The data usually comes in 1/100 form (percentages), so divide by 100
        # Headers might be 'Mkt-RF', 'SMB', 'HML', 'RF' for 3-factor or 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF' for 5-factor
        ff_factors = pd.read_csv(data_io, sep=r'\s+', header=0, index_col=0, parse_dates=True)

        # Convert index to datetime objects, handling various formats
        ff_factors.index = pd.to_datetime(ff_factors.index, format='%Y%m%d', errors='coerce')

        # Drop rows where date conversion failed
        ff_factors = ff_factors.dropna(subset=[ff_factors.index.name])

        # Filter by date range
        ff_factors = ff_factors[(ff_factors.index >= start_date) & (ff_factors.index <= end_date)]

        # Standardize column names (some files use Mkt-RF, others Mkt_Rf, etc.)
        ff_factors.columns = ff_factors.columns.str.replace('-', '_').str.strip()

        # Divide by 100 as factor data is usually in percentages
        numeric_cols = ff_factors.select_dtypes(include=[np.number]).columns
        ff_factors[numeric_cols] = ff_factors[numeric_cols] / 100

        print("Fama-French factor data download and initial processing complete.")
        return ff_factors

    except requests.exceptions.RequestException as e:
        print(f"Error downloading Fama-French data: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error processing Fama-French data: {e}")
        return pd.DataFrame()

# URL for daily Fama-French 5 Factors (Adjust if 3 factors or a different period is needed)
# Check the latest URL from Kenneth French's website if this one is outdated.
ff_url_5_factors = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_Daily.CSV"
ff_url_3_factors = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_Daily.CSV"

# We will prioritize 5 factors, but fall back to 3 if 5-factor download fails
fama_french_factors = download_fama_french_factors(ff_url_5_factors, START_DATE, END_DATE)

if fama_french_factors.empty:
    print("Attempting to download 3-Factor data as 5-Factor download failed or is empty.")
    fama_french_factors = download_fama_french_factors(ff_url_3_factors, START_DATE, END_DATE)

if not fama_french_factors.empty:
    print("\n--- Fama-French Factors (first 5 rows) ---")
    display(fama_french_factors.head())
    print("Columns:", fama_french_factors.columns.tolist())
else:
    print("Warning: Fama-French factor data could not be downloaded or is empty.")


### 8.3 Risk-Free Rate (FRED API or from Fama-French Data)

The Fama-French data usually includes the risk-free rate (`RF`). We will extract it from there. If it wasn't available, we would use the `fredapi` library to fetch it.

In [ ]:
if 'RF' in fama_french_factors.columns:
    risk_free_rate = fama_french_factors[['RF']].copy()
    print("Risk-free rate extracted from Fama-French data.")
elif 'RF' not in fama_french_factors.columns and 'fredapi' in globals(): # Check if fredapi was imported
    try:
        from fredapi import Fred
        # If FRED_API_KEY is defined in config, use it. Otherwise, may rely on public access (limited).
        fred = Fred(api_key=FRED_API_KEY if 'FRED_API_KEY' in globals() else None)
        print("Downloading risk-free rate from FRED (3-Month Treasury Bill Secondary Market Rate)...")
        # Example: Daily 3-Month Treasury Bill Secondary Market Rate
        risk_free_rate = fred.get_series('DTB3', observation_start=START_DATE, observation_end=END_DATE)
        risk_free_rate = risk_free_rate.to_frame(name='RF') / 100 / 365 # Convert to daily and decimal
        risk_free_rate.index = pd.to_datetime(risk_free_rate.index)
        print("Risk-free rate downloaded from FRED.")
    except Exception as e:
        print(f"Warning: Could not download risk-free rate from FRED: {e}")
        risk_free_rate = pd.DataFrame()
else:
    print("Warning: Risk-free rate (RF) not found in Fama-French data and FRED API key not configured or fredapi not installed/imported. Proceeding without RF.")
    risk_free_rate = pd.DataFrame()

if not risk_free_rate.empty:
    print("\n--- Risk-Free Rate (first 5 rows) ---")
    display(risk_free_rate.head())
else:
    print("Risk-free rate data is empty.")


## 9. Data Cleaning & Feature Engineering Implementation

### 9.1 Calculate Returns and Merge Data

This section calculates daily logarithmic returns for individual stocks and the market, and then merges them with the Fama-French factors and risk-free rate. Missing values will be handled by forward-filling or dropping.

In [ ]:
def calculate_log_returns(df):
    """
    Calculates daily logarithmic returns for a DataFrame of prices.
    """
    return np.log(df / df.shift(1))

# Calculate log returns for stocks
stock_returns = calculate_log_returns(stock_prices)

# Calculate log returns for market
market_returns = calculate_log_returns(market_prices)

# Merge stock returns, market returns, Fama-French factors, and risk-free rate
# We will use an outer join to ensure we capture all dates, then filter.

# Start with stock returns
merged_data = stock_returns.copy()

# Merge market returns
if not market_returns.empty:
    merged_data = merged_data.merge(market_returns, left_index=True, right_index=True, how='outer')

# Merge Fama-French factors
if not fama_french_factors.empty:
    merged_data = merged_data.merge(fama_french_factors, left_index=True, right_index=True, how='outer')

# Merge risk-free rate
if not risk_free_rate.empty:
    merged_data = merged_data.merge(risk_free_rate, left_index=True, right_index=True, how='outer')

# Sort by index (date)
merged_data = merged_data.sort_index()

# Forward-fill any missing data that might occur due to non-trading days or slight data discrepancies
# Then drop rows with any remaining NaN values (e.g., at the very beginning where returns can't be calculated)
merged_data = merged_data.ffill()
merged_data = merged_data.dropna()

# Ensure all columns are numeric
for col in merged_data.columns:
    merged_data[col] = pd.to_numeric(merged_data[col], errors='coerce')
merged_data = merged_data.dropna()

print("\n--- Merged and Cleaned Data (first 5 rows) ---")
display(merged_data.head())
print(f"Total data points after cleaning: {len(merged_data)}")
print(f"Columns in merged data: {merged_data.columns.tolist()}")


### 9.2 Calculate Excess Returns

For factor models, we typically work with excess returns (asset return minus risk-free rate) and market excess returns (market return minus risk-free rate). The Fama-French factors (Mkt-Rf, SMB, HML, RMW, CMA) are already excess returns, so we only need to calculate excess returns for our individual stocks and ensure `Mkt_Rf` is present.

In [ ]:
# Identify all stock columns dynamically
stock_columns = [col for col in TICKERS if col in merged_data.columns]

# Calculate excess returns for individual stocks
if 'RF' in merged_data.columns:
    for col in stock_columns:
        merged_data[f'{col}_Excess_Return'] = merged_data[col] - merged_data['RF']
    # Ensure Mkt_Rf is correctly used or calculated if 'Market' is available and 'Mkt_Rf' is not
    if 'Mkt_Rf' not in merged_data.columns and 'Market' in merged_data.columns:
        merged_data['Mkt_Rf'] = merged_data['Market'] - merged_data['RF']
    elif 'Mkt_Rf' not in merged_data.columns and 'Market' not in merged_data.columns:
        print("Warning: Neither 'Mkt_Rf' nor 'Market' columns found in merged data to calculate market excess return.")
else:
    print("Warning: Risk-free rate (RF) not found in merged data. Cannot calculate excess returns.")

# Display the first few rows with excess returns
print("\n--- Merged Data with Excess Returns (first 5 rows) ---")
display(merged_data.filter(like='Excess_Return').head())
display(merged_data[['Mkt_Rf', 'SMB', 'HML', 'RMW', 'CMA']].head() if all(f in merged_data.columns for f in ['Mkt_Rf', 'SMB', 'HML', 'RMW', 'CMA']) else merged_data[['Mkt_Rf', 'SMB', 'HML']].head())

# Store the processed data for later use
processed_data = merged_data.copy()

print(f"Data cleaning and feature engineering complete. Processed data has {processed_data.shape[0]} rows and {processed_data.shape[1]} columns.")


## 10. Core Models/Algorithms: Factor Estimation Implementation

### 10.1 Rolling Multi-Factor Regression

This section implements rolling ordinary least squares (OLS) regression to estimate the factor exposures (betas) for each stock. The Fama-French factors are used as independent variables and the stock's excess return as the dependent variable. A common rolling window (e.g., 60 or 120 days) is used to capture changes in factor sensitivities over time. The estimated betas are crucial inputs for portfolio optimization.

In [ ]:
def estimate_factor_exposures(processed_data, tickers, factor_names, rolling_window=120):
    """
    Estimates rolling factor exposures (betas) for a list of tickers
    using the Fama-French factors via RollingOLS regression.

    Args:
        processed_data (pd.DataFrame): DataFrame containing stock excess returns and Fama-French factors.
        tickers (list): List of stock ticker symbols.
        factor_names (list): List of Fama-French factor column names (e.g., ['Mkt_Rf', 'SMB', 'HML']).
        rolling_window (int): The window size for the rolling regression (in days).

    Returns:
        pd.DataFrame: DataFrame with rolling factor betas for each stock and factor.
                      Index is date, columns are in format 'TICKER_FACTOR'.
    """
    print(f"Estimating rolling factor exposures over a {rolling_window}-day window...")
    all_betas = []

    # Add a constant to the independent variables for the regression intercept (alpha)
    X = sm.add_constant(processed_data[factor_names].copy())

    for ticker in tickers:
        # Dependent variable: stock's excess return
        y_col = f'{ticker}_Excess_Return'
        if y_col not in processed_data.columns:
            print(f"Warning: {y_col} not found in processed_data. Skipping {ticker}.")
            continue

        y = processed_data[y_col]

        # Perform rolling OLS regression
        # Handle potential NaNs in y or X due to earlier processing/missing data
        temp_df = pd.concat([y, X], axis=1).dropna()

        if temp_df.empty or len(temp_df) < rolling_window:
            print(f"Not enough data for rolling regression for {ticker}. Skipping.")
            continue

        rols = RollingOLS(endog=temp_df[y_col], exog=temp_df[factor_names], window=rolling_window)
        rres = rols.fit()

        # Extract the betas (coefficients)
        # Note: 'const' is the alpha, factors are the betas
        betas = rres.params.copy()
        betas.columns = [f'{ticker}_{col}' for col in betas.columns]
        all_betas.append(betas)

    if not all_betas:
        print("No factor exposures could be estimated.")
        return pd.DataFrame()

    # Concatenate all betas into a single DataFrame
    factor_exposures = pd.concat(all_betas, axis=1)
    factor_exposures = factor_exposures.dropna()
    print("Factor exposure estimation complete.")
    return factor_exposures

# Dynamically determine available Fama-French factors
# Exclude 'RF' and the market excess return itself if it's already a factor
# Mkt_Rf is the market factor, so we keep it.
ff_factor_names = [col for col in fama_french_factors.columns if col not in ['RF']]

# Estimate factor exposures (betas)
# Using a rolling window of 120 days (approx 6 months of trading days)
rolling_window = 120
factor_exposures = estimate_factor_exposures(processed_data, TICKERS, ff_factor_names, rolling_window)

if not factor_exposures.empty:
    print(f"\n--- Estimated Factor Exposures (first 5 rows for a few stocks) ---")
    # Display first few columns for brevity
    display(factor_exposures.iloc[:, :min(5, factor_exposures.shape[1])].head())
    print(f"Factor exposures estimated for {len(factor_exposures.columns) // len(ff_factor_names)} stocks.")
else:
    print("Warning: Factor exposures could not be estimated. Check data or parameters.")


### 10.2 Preparing Data for Portfolio Optimization

For portfolio optimization, we need expected returns, expected covariance matrix of returns, and factor exposures at each rebalancing point. Since we are doing monthly rebalancing, we will resample the daily data to monthly frequency to get monthly returns and take the last available factor exposures for each month-end. We also need to handle the risk-free rate for the optimization.

In [ ]:
# Resample daily excess returns to monthly
# Use mean of daily returns for monthly return (approximation for log returns)
monthly_excess_returns = processed_data.filter(like='_Excess_Return').resample('M').apply(lambda x: (1 + x).prod() - 1)

# Align factor exposures to monthly frequency (take last available estimate of the month)
monthly_factor_exposures = factor_exposures.resample('M').last().dropna()

# Ensure tickers in monthly_excess_returns match those in monthly_factor_exposures
# And that both have enough data points

# Filter monthly_excess_returns to only include tickers for which we have factor exposures
filtered_tickers = [col.replace('_Excess_Return', '') for col in monthly_excess_returns.columns
                    if f"{col.replace('_Excess_Return', '')}_{ff_factor_names[0]}" in monthly_factor_exposures.columns]

monthly_excess_returns_filtered = monthly_excess_returns[[f'{t}_Excess_Return' for t in filtered_tickers]]

# Prepare monthly risk-free rate (if available) for Sharpe ratio calculations
# Assuming RF is daily, resample to monthly and convert to monthly rate
if 'RF' in processed_data.columns:
    monthly_rf = processed_data['RF'].resample('M').apply(lambda x: (1 + x).prod() - 1)
    monthly_rf.name = 'Monthly_RF'
    monthly_rf = monthly_rf.dropna()
else:
    monthly_rf = pd.Series(0, index=monthly_excess_returns_filtered.index) # Assume 0 if not available
    print("Warning: Monthly risk-free rate not available. Using 0 for optimization.")


print("\n--- Monthly Excess Returns (first 5 rows) ---")
display(monthly_excess_returns_filtered.head())
print("\n--- Monthly Factor Exposures (first 5 rows) ---")
display(monthly_factor_exposures.iloc[:, :min(5, monthly_factor_exposures.shape[1])].head())
print("\n--- Monthly Risk-Free Rate (first 5 rows) ---")
display(monthly_rf.head())

print(f"Prepared monthly data for {len(filtered_tickers)} stocks from {monthly_excess_returns_filtered.index.min().strftime('%Y-%m')} to {monthly_excess_returns_filtered.index.max().strftime('%Y-%m')}.")


## 10.3 Portfolio Optimization Implementation (Monthly Rebalancing)

This section implements the core portfolio optimization logic using `PyPortfolioOpt`. The process will simulate a monthly rebalancing strategy, where at the end of each month, we calculate the expected returns and covariance matrix based on historical data up to that point, and then optimize portfolio weights. For simplicity, we will start with a Maximum Sharpe Ratio optimization, allowing long-only positions.

In [ ]:
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
from pypfopt import HRPOpt # For Hierarchical Risk Parity (alternative)

def run_monthly_optimization(monthly_excess_returns, monthly_factor_exposures, monthly_rf,
                           ff_factor_names, optimization_strategy='max_sharpe',
                           lookback_period_years_for_returns=5,
                           lookback_period_years_for_cov=5):
    """
    Performs monthly portfolio optimization based on a chosen strategy.

    Args:
        monthly_excess_returns (pd.DataFrame): Monthly excess returns for all stocks.
        monthly_factor_exposures (pd.DataFrame): Monthly factor exposures (betas).
        monthly_rf (pd.Series): Monthly risk-free rate.
        ff_factor_names (list): List of Fama-French factor names.
        optimization_strategy (str): 'max_sharpe' or 'hrp'.
        lookback_period_years_for_returns (int): Years of data to use for expected returns.
        lookback_period_years_for_cov (int): Years of data to use for covariance matrix.

    Returns:
        pd.DataFrame: DataFrame of optimal weights for each rebalancing period.
    """
    print(f"Starting monthly portfolio optimization using {optimization_strategy.replace('_', ' ').title()} strategy...")

    # Convert monthly excess returns back to total returns for PyPortfolioOpt
    # Assuming monthly_excess_returns are already 'excess', so add RF if available
    if not monthly_rf.empty:
        total_monthly_returns = monthly_excess_returns.add(monthly_rf, axis=0)
    else:
        total_monthly_returns = monthly_excess_returns # If no RF, excess = total
        print("Warning: Risk-free rate not available for converting to total returns. Using excess returns as total.")

    # Ensure the total_monthly_returns index is datetime for resampling
    total_monthly_returns.index = pd.to_datetime(total_monthly_returns.index)
    monthly_factor_exposures.index = pd.to_datetime(monthly_factor_exposures.index)
    if not monthly_rf.empty: monthly_rf.index = pd.to_datetime(monthly_rf.index)

    # Get all unique tickers from the monthly excess returns
    all_tickers = [col.replace('_Excess_Return', '') for col in monthly_excess_returns.columns]

    optimal_weights = pd.DataFrame(columns=all_tickers)
    rebalance_dates = []

    # Iterate through each rebalancing point (end of month)
    for i in range(len(monthly_excess_returns.index)):
        current_date = monthly_excess_returns.index[i]
        rebalance_dates.append(current_date)

        # Define lookback period for expected returns and covariance
        returns_lookback_start = current_date - pd.DateOffset(years=lookback_period_years_for_returns)
        cov_lookback_start = current_date - pd.DateOffset(years=lookback_period_years_for_cov)

        # Data for expected returns and covariance (use data *before* current_date)
        returns_history = total_monthly_returns.loc[returns_lookback_start:current_date - pd.DateOffset(days=1)].dropna(axis=1, how='all')

        # Filter for tickers that have sufficient data in returns_history and also have factor exposures for current_date
        active_tickers_returns = [col for col in returns_history.columns if not returns_history[col].isnull().all()]

        # Filter factor exposures to only include active tickers for the current month
        current_factor_exposures = monthly_factor_exposures.loc[current_date].dropna() if current_date in monthly_factor_exposures.index else pd.Series()
        active_tickers_factors = [col.split('_')[0] for col in current_factor_exposures.index if any(f in col for f in ff_factor_names)]

        # Final set of investable assets for this period
        investable_assets = sorted(list(set(active_tickers_returns) & set(active_tickers_factors)))

        if len(investable_assets) < 2: # Need at least 2 assets for most optimizations
            print(f"Not enough investable assets ({len(investable_assets)}) for optimization at {current_date.strftime('%Y-%m')}. Skipping.")
            optimal_weights.loc[current_date] = 0.0
            continue

        # Subset returns history to only include investable assets
        returns_history_filtered = returns_history[investable_assets]

        # Skip if filtered history is empty or not enough rows for lookback
        if returns_history_filtered.empty or len(returns_history_filtered) < 12 * lookback_period_years_for_cov / 2: # At least half of expected data points
             print(f"Not enough historical return data for investable assets at {current_date.strftime('%Y-%m')}. Skipping.")
             optimal_weights.loc[current_date] = 0.0
             continue

        try:
            # 1. Expected Returns
            mu = expected_returns.mean_historical_return(returns_history_filtered, frequency=12)

            # 2. Covariance Matrix
            S = risk_models.sample_cov(returns_history_filtered, frequency=12)

            # Ensure mu and S are aligned (same tickers in same order)
            common_assets = list(mu.index.intersection(S.columns))
            if len(common_assets) < len(mu) or len(common_assets) < len(S):
                mu = mu.loc[common_assets]
                S = S.loc[common_assets, common_assets]

            if S.empty or mu.empty or len(S) < 2:
                print(f"Covariance or Expected Returns empty/insufficient for {current_date}. Skipping.")
                optimal_weights.loc[current_date] = 0.0
                continue

            # 3. Optimization
            if optimization_strategy == 'max_sharpe':
                # Risk-free rate for Sharpe ratio calculation (take monthly_rf for this date if available)
                current_rf = monthly_rf.loc[current_date] if not monthly_rf.empty and current_date in monthly_rf.index else 0.0
                ef = EfficientFrontier(mu, S, weight_bounds=(0, 1)) # Long-only
                raw_weights = ef.max_sharpe(risk_free_rate=current_rf)
                cleaned_weights = ef.clean_weights()

            elif optimization_strategy == 'hrp':
                hrp = HRPOpt(returns_history_filtered)
                hrp.optimize()
                cleaned_weights = hrp.clean_weights()

            else:
                raise ValueError("Invalid optimization strategy.")

            # Store weights
            current_weights = pd.Series(cleaned_weights)
            optimal_weights.loc[current_date] = current_weights

        except Exception as e:
            print(f"Error optimizing portfolio for {current_date.strftime('%Y-%m')}: {e}. Setting weights to zero.")
            optimal_weights.loc[current_date] = 0.0

    # Fill any remaining NaNs (for dates where optimization was skipped) with 0
    optimal_weights = optimal_weights.fillna(0.0)
    # Align columns to the full set of tickers, adding 0 for any missing
    optimal_weights = optimal_weights.reindex(columns=all_tickers, fill_value=0.0)
    optimal_weights.index.name = 'Date'
    print("Monthly portfolio optimization complete.")
    return optimal_weights

# Run the optimization
# It's crucial that `monthly_excess_returns_filtered` contains _only_ the tickers we want to optimize.
# Make sure `ff_factor_names` matches the columns used for factor estimation earlier.

optimal_portfolio_weights = run_monthly_optimization(
    monthly_excess_returns_filtered,
    monthly_factor_exposures,
    monthly_rf,
    ff_factor_names,
    optimization_strategy='max_sharpe'
)

if not optimal_portfolio_weights.empty:
    print("\n--- Optimal Portfolio Weights (first 5 months) ---")
    display(optimal_portfolio_weights.head())
    print("\n--- Optimal Portfolio Weights (last 5 months) ---")
    display(optimal_portfolio_weights.tail())
    print(f"Total rebalancing periods: {len(optimal_portfolio_weights)}")
else:
    print("Warning: No optimal portfolio weights could be calculated.")


## 11. Backtesting & Performance Analysis

### 11.1 Portfolio Backtesting

This section simulates the performance of the optimized portfolio using the calculated monthly weights. We will compute the portfolio's monthly returns based on these weights and the actual stock returns during each period. We'll also calculate the benchmark (SPY) returns for comparison.

In [ ]:
def backtest_portfolio(optimal_weights, monthly_total_returns, benchmark_returns, monthly_rf):
    """
    Simulates the portfolio's performance based on optimal weights.

    Args:
        optimal_weights (pd.DataFrame): DataFrame of optimal weights for each rebalancing period.
        monthly_total_returns (pd.DataFrame): Monthly total returns for all stocks.
        benchmark_returns (pd.Series): Monthly returns for the benchmark.
        monthly_rf (pd.Series): Monthly risk-free rate.

    Returns:
        pd.DataFrame: DataFrame containing portfolio returns, benchmark returns, and other metrics.
    """
    print("Starting portfolio backtesting...")

    # Ensure indices are aligned and are datetime objects
    optimal_weights.index = pd.to_datetime(optimal_weights.index)
    monthly_total_returns.index = pd.to_datetime(monthly_total_returns.index)
    benchmark_returns.index = pd.to_datetime(benchmark_returns.index)
    monthly_rf.index = pd.to_datetime(monthly_rf.index)

    portfolio_returns = pd.Series(dtype=float)
    portfolio_excess_returns = pd.Series(dtype=float)

    # Align all dataframes by date
    # Use inner join on dates to ensure all components are present for a given month
    start_date = max(optimal_weights.index.min(), monthly_total_returns.index.min(),
                     benchmark_returns.index.min(), monthly_rf.index.min())
    end_date = min(optimal_weights.index.max(), monthly_total_returns.index.max(),
                   benchmark_returns.index.max(), monthly_rf.index.max())

    # Filter data to common date range
    optimal_weights_aligned = optimal_weights.loc[start_date:end_date].copy()
    monthly_total_returns_aligned = monthly_total_returns.loc[start_date:end_date].copy()
    benchmark_returns_aligned = benchmark_returns.loc[start_date:end_date].copy()
    monthly_rf_aligned = monthly_rf.loc[start_date:end_date].copy()

    # Iterate through each rebalancing month (except the very first one, which is just for initial weights)
    # We need weights from the *previous* month to apply to the *current* month's returns.
    for i in range(1, len(optimal_weights_aligned)):
        rebalance_date = optimal_weights_aligned.index[i]
        prev_rebalance_date = optimal_weights_aligned.index[i-1]

        # Get weights from the previous period (end of month)
        current_weights = optimal_weights_aligned.loc[prev_rebalance_date]

        # Get actual returns for the current month
        # We need returns for the period *after* the weights were set.
        # This requires careful index alignment. Monthly returns are typically for the period *ending* on that date.
        # So, weights at month-end `t-1` apply to returns from `t-1` to `t`.
        # The `monthly_total_returns` are indexed by the end of the month for which the return is observed.
        current_month_returns = monthly_total_returns_aligned.loc[rebalance_date]

        # Filter current_month_returns to only include assets for which we have weights
        # And ensure weights and returns have the same asset universe and order
        common_assets = current_weights.index.intersection(current_month_returns.index)

        if common_assets.empty:
            portfolio_monthly_return = 0.0
        else:
            current_weights_filtered = current_weights.loc[common_assets].fillna(0)
            current_month_returns_filtered = current_month_returns.loc[common_assets].fillna(0)

            # Normalize weights if they don't sum to 1 (e.g., due to filtering out assets)
            if current_weights_filtered.sum() > 0:
                 current_weights_filtered = current_weights_filtered / current_weights_filtered.sum()
            else: # If all weights are zero after filtering, portfolio return is zero
                 portfolio_monthly_return = 0.0
                 portfolio_returns.loc[rebalance_date] = portfolio_monthly_return
                 portfolio_excess_returns.loc[rebalance_date] = portfolio_monthly_return - monthly_rf_aligned.loc[rebalance_date]
                 continue

            # Calculate portfolio return for the month
            portfolio_monthly_return = (current_weights_filtered * current_month_returns_filtered).sum()

        portfolio_returns.loc[rebalance_date] = portfolio_monthly_return
        portfolio_excess_returns.loc[rebalance_date] = portfolio_monthly_return - monthly_rf_aligned.loc[rebalance_date]

    # Combine results
    performance_df = pd.DataFrame({
        'Portfolio_Return': portfolio_returns,
        'Benchmark_Return': benchmark_returns_aligned,
        'Portfolio_Excess_Return': portfolio_excess_returns,
        'Benchmark_Excess_Return': benchmark_returns_aligned - monthly_rf_aligned
    }).dropna()

    print("Portfolio backtesting complete.")
    return performance_df


# Prepare monthly total returns from processed_data for backtesting
# This needs to be done carefully to ensure correct alignment.
# We previously calculated monthly_excess_returns_filtered which were log returns converted to simple.
# Let's ensure we have simple *total* returns here for accurate backtesting.

# First, re-calculate daily simple returns for all stocks and market
daily_simple_returns = stock_prices.pct_change().dropna()
daily_market_simple_returns = market_prices.pct_change().dropna().squeeze() # Squeeze to series

# Then, resample to monthly total returns
# For simple returns, (1+R1)(1+R2)... - 1
monthly_simple_returns_stocks = (1 + daily_simple_returns).resample('M').prod() - 1
monthly_simple_returns_market = (1 + daily_market_simple_returns).resample('M').prod() - 1

# Ensure monthly_simple_returns_stocks only contains the tickers for which we optimized
filtered_tickers_for_backtest = [col for col in optimal_portfolio_weights.columns if col in monthly_simple_returns_stocks.columns]
monthly_simple_returns_stocks = monthly_simple_returns_stocks[filtered_tickers_for_backtest]

# Align monthly_rf to the same index as monthly_simple_returns_stocks
monthly_rf_for_backtest = (1 + risk_free_rate).resample('M').prod() - 1
monthly_rf_for_backtest = monthly_rf_for_backtest.reindex(monthly_simple_returns_stocks.index, method='ffill')

# Run the backtest
performance_results = backtest_portfolio(
    optimal_portfolio_weights,
    monthly_simple_returns_stocks,
    monthly_simple_returns_market,
    monthly_rf_for_backtest.squeeze() # Ensure it's a Series
)

if not performance_results.empty:
    print("\n--- Backtest Performance Results (first 5 months) ---")
    display(performance_results.head())
    print("\n--- Backtest Performance Results (last 5 months) ---")
    display(performance_results.tail())
    print(f"Backtested over {len(performance_results)} months.")
else:
    print("Warning: No performance results generated. Check input data and alignment.")


### 11.2 Performance Metrics

This section calculates key performance indicators for both the portfolio and the benchmark, such as annualized returns, volatility, Sharpe ratio, and Maximum Drawdown. These metrics provide a comprehensive view of the strategy's effectiveness and risk-adjusted returns.

In [ ]:
def calculate_performance_metrics(returns_series, annual_factor=12):
    """
    Calculates annualized returns, volatility, and Sharpe Ratio.

    Args:
        returns_series (pd.Series): Series of monthly returns.
        annual_factor (int): Factor to annualize (12 for monthly).

    Returns:
        dict: Dictionary of performance metrics.
    """
    if returns_series.empty or returns_series.isnull().all():
        return {
            'Annualized Return': np.nan,
            'Annualized Volatility': np.nan,
            'Sharpe Ratio': np.nan,
            'Maximum Drawdown': np.nan
        }

    # Cumulative returns
    cumulative_returns = (1 + returns_series).cumprod()

    # Annualized Return
    total_months = len(returns_series)
    annualized_return = (cumulative_returns.iloc[-1])**(annual_factor / total_months) - 1

    # Annualized Volatility
    annualized_volatility = returns_series.std() * np.sqrt(annual_factor)

    # Sharpe Ratio (using risk-free rate from the portfolio's period for excess returns)
    # If we have excess returns directly, use them. Otherwise, compute.
    # For this function, let's assume `returns_series` is already excess returns if Sharpe is needed
    # Or, provide RF separately. For now, we'll calculate it for total returns, assuming RF is zero if not specified.

    # Maximum Drawdown
    peak = cumulative_returns.cummax()
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = drawdown.min()

    return {
        'Annualized Return': annualized_return,
        'Annualized Volatility': annualized_volatility,
        # Sharpe Ratio requires excess returns. Will calculate outside if RF is known.
        'Maximum Drawdown': max_drawdown
    }

print("Calculating performance metrics...")

# Calculate metrics for Portfolio
portfolio_metrics = calculate_performance_metrics(performance_results['Portfolio_Return'])

# Calculate metrics for Benchmark
benchmark_metrics = calculate_performance_metrics(performance_results['Benchmark_Return'])

# Calculate Sharpe Ratios using excess returns
# Need to ensure monthly_rf is aligned with performance_results for this.
# Assuming `monthly_rf_for_backtest` is the correct aligned risk-free rate.

# Filter monthly_rf_for_backtest to the same index as performance_results
aligned_monthly_rf = monthly_rf_for_backtest.reindex(performance_results.index, method='ffill').squeeze()

if not aligned_monthly_rf.empty:
    # Portfolio Sharpe
    portfolio_excess_returns = performance_results['Portfolio_Return'] - aligned_monthly_rf
    portfolio_sharpe = portfolio_excess_returns.mean() / portfolio_excess_returns.std() * np.sqrt(12)
    portfolio_metrics['Sharpe Ratio'] = portfolio_sharpe

    # Benchmark Sharpe
    benchmark_excess_returns = performance_results['Benchmark_Return'] - aligned_monthly_rf
    benchmark_sharpe = benchmark_excess_returns.mean() / benchmark_excess_returns.std() * np.sqrt(12)
    benchmark_metrics['Sharpe Ratio'] = benchmark_sharpe
else:
    print("Warning: Risk-free rate not available for Sharpe Ratio calculation.")


performance_summary = pd.DataFrame({
    'Portfolio': portfolio_metrics,
    'Benchmark': benchmark_metrics
})

print("\n--- Performance Summary ---")
display(performance_summary)


### 11.3 Attribution Analysis (CAPM)

This section performs a basic Capital Asset Pricing Model (CAPM) attribution to decompose the portfolio's excess returns into market risk premium and idiosyncratic (alpha) components. A multi-factor attribution can be done if factors beyond market are used.

In [ ]:
def capm_attribution(portfolio_excess_returns, market_excess_returns):
    """
    Performs CAPM regression to get Alpha and Beta.

    Args:
        portfolio_excess_returns (pd.Series): Portfolio excess returns.
        market_excess_returns (pd.Series): Market excess returns.

    Returns:
        dict: Alpha, Beta, and R-squared from CAPM.
    """
    # Ensure indices are aligned
    common_index = portfolio_excess_returns.index.intersection(market_excess_returns.index)
    y = portfolio_excess_returns.loc[common_index].dropna()
    x = market_excess_returns.loc[common_index].dropna()

    if y.empty or x.empty:
        return {'Alpha': np.nan, 'Beta': np.nan, 'R-squared': np.nan}

    # Add a constant for Alpha
    X_const = sm.add_constant(x)

    try:
        model = sm.OLS(y, X_const)
        results = model.fit()
        alpha = results.params['const'] * 12 # Annualize alpha
        beta = results.params[market_excess_returns.name]
        r_squared = results.rsquared
    except Exception as e:
        print(f"Error during CAPM regression: {e}")
        alpha, beta, r_squared = np.nan, np.nan, np.nan

    return {'Alpha': alpha, 'Beta': beta, 'R-squared': r_squared}

print("Performing CAPM attribution...")

# Ensure market_excess_returns for CAPM is from the `performance_results` DataFrame
capm_results = capm_attribution(
    performance_results['Portfolio_Excess_Return'],
    performance_results['Benchmark_Excess_Return'] # Assuming 'Benchmark_Excess_Return' is the market excess for CAPM
)

print("\n--- CAPM Attribution Results (Annualized Alpha) ---")
for key, value in capm_results.items():
    if key == 'Alpha':
        print(f"{key}: {value:.4f} (Annualized)")
    else:
        print(f"{key}: {value:.4f}")


### 11.4 Multi-Factor Attribution

This provides a more detailed breakdown of returns attributable to each Fama-French factor, beyond just market exposure. This is performed by regressing the portfolio's excess returns against the Fama-French factors.

In [ ]:
def multi_factor_attribution(portfolio_excess_returns, fama_french_factors_for_attribution, factor_names):
    """
    Performs multi-factor regression for attribution.

    Args:
        portfolio_excess_returns (pd.Series): Portfolio excess returns.
        fama_french_factors_for_attribution (pd.DataFrame): DataFrame of Fama-French factors.
        factor_names (list): List of Fama-French factor column names.

    Returns:
        dict: Annualized Alpha, Betas for each factor, and R-squared.
    """
    # Ensure indices are aligned
    common_index = portfolio_excess_returns.index.intersection(fama_french_factors_for_attribution.index)
    y = portfolio_excess_returns.loc[common_index].dropna()
    X = sm.add_constant(fama_french_factors_for_attribution[factor_names].loc[common_index].dropna())

    if y.empty or X.empty or len(y) != len(X):
        return {'Alpha': np.nan, **{f'Beta_{f}': np.nan for f in factor_names}, 'R-squared': np.nan}

    try:
        model = sm.OLS(y, X)
        results = model.fit()

        attribution_results = {
            'Alpha': results.params['const'] * 12, # Annualize alpha
            **{f'Beta_{f}': results.params[f] for f in factor_names},
            'R-squared': results.rsquared
        }
    except Exception as e:
        print(f"Error during multi-factor regression: {e}")
        attribution_results = {'Alpha': np.nan, **{f'Beta_{f}': np.nan for f in factor_names}, 'R-squared': np.nan}

    return attribution_results

print("Performing Multi-Factor attribution...")

# Prepare Fama-French factors for attribution (monthly)
# Take monthly average of daily factors
monthly_ff_factors_for_attribution = fama_french_factors[ff_factor_names].resample('M').mean().dropna()

multi_factor_results = multi_factor_attribution(
    performance_results['Portfolio_Excess_Return'],
    monthly_ff_factors_for_attribution,
    ff_factor_names
)

print("\n--- Multi-Factor Attribution Results (Annualized Alpha & Betas) ---")
for key, value in multi_factor_results.items():
    if 'Alpha' in key:
        print(f"{key}: {value:.4f} (Annualized)")
    elif 'Beta' in key:
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value:.4f}")


## 12. Visualizations & Dashboard Components

This section generates professional and interactive visualizations to present the results of the factor investing engine. Visualizations are key for understanding performance, risk, and attribution.

### 12.1 Cumulative Returns Comparison

Compares the cumulative returns of the factor-optimized portfolio against the benchmark (SPY).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

def plot_cumulative_returns(performance_df, title='Cumulative Returns: Portfolio vs. Benchmark'):
    """
    Plots the cumulative returns of the portfolio and benchmark.
    """
    print("Generating cumulative returns plot...")
    cumulative_portfolio = (1 + performance_df['Portfolio_Return']).cumprod()
    cumulative_benchmark = (1 + performance_df['Benchmark_Return']).cumprod()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=cumulative_portfolio.index, y=cumulative_portfolio, mode='lines', name='Factor Portfolio'))
    fig.add_trace(go.Scatter(x=cumulative_benchmark.index, y=cumulative_benchmark, mode='lines', name='Benchmark (SPY)'))

    fig.update_layout(
        title_text=title,
        xaxis_title='Date',
        yaxis_title='Cumulative Return',
        hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        template='plotly_white'
    )
    fig.show()

if not performance_results.empty:
    plot_cumulative_returns(performance_results)
else:
    print("Cannot plot cumulative returns: Performance results are empty.")

### 12.2 Rolling Performance Metrics

Visualizes rolling annualized returns and rolling Sharpe Ratios to assess the consistency of performance over time.

In [ ]:
def plot_rolling_performance(performance_df, window_size=24, annual_factor=12):
    """
    Plots rolling annualized returns and rolling Sharpe Ratios.

    Args:
        performance_df (pd.DataFrame): DataFrame with portfolio and benchmark returns.
        window_size (int): Rolling window size in months.
        annual_factor (int): Factor to annualize (12 for monthly).
    """
    print(f"Generating rolling performance plots (window_size={window_size} months)...")

    rolling_portfolio_ret = (1 + performance_df['Portfolio_Return']).rolling(window=window_size).apply(lambda x: x.prod())**(annual_factor/window_size) - 1
    rolling_benchmark_ret = (1 + performance_df['Benchmark_Return']).rolling(window=window_size).apply(lambda x: x.prod())**(annual_factor/window_size) - 1

    rolling_portfolio_vol = performance_df['Portfolio_Return'].rolling(window=window_size).std() * np.sqrt(annual_factor)
    rolling_benchmark_vol = performance_df['Benchmark_Return'].rolling(window=window_size).std() * np.sqrt(annual_factor)

    # Rolling Sharpe Ratio (using rolling excess returns)
    rolling_portfolio_sharpe = (performance_df['Portfolio_Excess_Return'].rolling(window=window_size).mean() / performance_df['Portfolio_Excess_Return'].rolling(window=window_size).std()) * np.sqrt(annual_factor)
    rolling_benchmark_sharpe = (performance_df['Benchmark_Excess_Return'].rolling(window=window_size).mean() / performance_df['Benchmark_Excess_Return'].rolling(window=window_size).std()) * np.sqrt(annual_factor)

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(f'{window_size}-Month Rolling Annualized Returns',
                                        f'{window_size}-Month Rolling Sharpe Ratio'))

    # Rolling Returns
    fig.add_trace(go.Scatter(x=rolling_portfolio_ret.index, y=rolling_portfolio_ret, mode='lines', name='Portfolio Returns', legendgroup='returns', legendgrouptitle_text='Returns'), row=1, col=1)
    fig.add_trace(go.Scatter(x=rolling_benchmark_ret.index, y=rolling_benchmark_ret, mode='lines', name='Benchmark Returns', legendgroup='returns', legendgrouptitle_text='Returns'), row=1, col=1)
    fig.update_yaxes(title_text='Annualized Return', row=1, col=1)

    # Rolling Sharpe Ratio
    fig.add_trace(go.Scatter(x=rolling_portfolio_sharpe.index, y=rolling_portfolio_sharpe, mode='lines', name='Portfolio Sharpe', legendgroup='sharpe', legendgrouptitle_text='Sharpe Ratio'), row=2, col=1)
    fig.add_trace(go.Scatter(x=rolling_benchmark_sharpe.index, y=rolling_benchmark_sharpe, mode='lines', name='Benchmark Sharpe', legendgroup='sharpe', legendgrouptitle_text='Sharpe Ratio'), row=2, col=1)
    fig.update_yaxes(title_text='Sharpe Ratio', row=2, col=1)

    fig.update_layout(height=800, title_text=f'Rolling Performance Metrics ({window_size}-Month)', template='plotly_white', hovermode='x unified')
    fig.show()

if not performance_results.empty:
    plot_rolling_performance(performance_results, window_size=24)
else:
    print("Cannot plot rolling performance: Performance results are empty.")

### 12.3 Factor Exposures Over Time

Illustrates how the portfolio's exposure to different Fama-French factors changes over the backtesting period. This helps understand the dynamic nature of the factor strategy.

In [ ]:
def plot_factor_exposures(factor_exposures_df, ff_factor_names):
    """
    Plots the average factor exposures of the portfolio over time.
    """
    print("Generating factor exposures over time plot...")

    # Calculate the average exposure to each factor across all stocks in the portfolio at each time step
    # This is an approximation as it doesn't weight by portfolio weight for now.
    # For a more precise plot, we'd multiply factor_exposures by optimal_weights and sum.
    # Let's average the individual stock exposures for each factor for simplicity here.

    # First, rename columns in factor_exposures_df to extract stock and factor names
    # We need to reshape the dataframe to easily average factor exposures
    reshaped_exposures = []
    for col in factor_exposures_df.columns:
        parts = col.rsplit('_', len(ff_factor_names))
        # The factor name is the last part if it matches a ff_factor_name
        factor = parts[-1]
        if factor in ff_factor_names:
            ticker = '_'.join(parts[:-1])
            reshaped_exposures.append({
                'Date': factor_exposures_df.index,
                'Ticker': ticker,
                'Factor': factor,
                'Exposure': factor_exposures_df[col].values
            })

    if not reshaped_exposures:
        print("No factor exposures to plot.")
        return

    # Convert list of dicts to DataFrame for easier manipulation
    df_exposures = pd.DataFrame({
        'Date': pd.concat([d['Date'] for d in reshaped_exposures]),
        'Ticker': pd.concat([pd.Series(d['Ticker'], index=d['Date']) for d in reshaped_exposures]),
        'Factor': pd.concat([pd.Series(d['Factor'], index=d['Date']) for d in reshaped_exposures]),
        'Exposure': pd.concat([pd.Series(d['Exposure'], index=d['Date']) for d in reshaped_exposures])
    })
    df_exposures['Date'] = pd.to_datetime(df_exposures['Date'])
    df_exposures = df_exposures.set_index('Date')

    # Calculate the monthly average exposure for each factor
    # We take the mean of all individual stock exposures to a factor
    monthly_avg_factor_exposures = df_exposures.groupby(['Date', 'Factor'])['Exposure'].mean().unstack()
    monthly_avg_factor_exposures = monthly_avg_factor_exposures.resample('M').mean().dropna(how='all')

    if monthly_avg_factor_exposures.empty:
        print("No monthly average factor exposures to plot.")
        return

    fig = go.Figure()
    for factor in monthly_avg_factor_exposures.columns:
        fig.add_trace(go.Scatter(x=monthly_avg_factor_exposures.index, y=monthly_avg_factor_exposures[factor], mode='lines', name=factor))

    fig.update_layout(
        title_text='Average Factor Exposures Over Time',
        xaxis_title='Date',
        yaxis_title='Exposure (Beta)',
        hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        template='plotly_white'
    )
    fig.show()

if not factor_exposures.empty and ff_factor_names:
    plot_factor_exposures(factor_exposures, ff_factor_names)
else:
    print("Cannot plot factor exposures: Factor exposure data or factor names are empty.")

### 12.4 Efficient Frontier (Illustrative)

An illustrative plot of the Efficient Frontier, showcasing the trade-off between risk and return, and where the optimized portfolio (or the benchmark) falls on this curve. Note that this is typically calculated using the *entire* historical period's returns, not rolling, so it serves as a retrospective illustration.

In [ ]:
from pypfopt import plotting

def plot_efficient_frontier(monthly_total_returns, monthly_rf, lookback_period_years=5):
    """
    Plots an illustrative efficient frontier based on historical returns.
    """
    print("Generating Efficient Frontier plot...")

    # Use the entire data for a general efficient frontier, or a fixed lookback for a 'current' view
    # For this illustrative plot, let's use a recent fixed lookback period for expected returns and covariance
    end_date_for_ef = monthly_total_returns.index.max()
    start_date_for_ef = end_date_for_ef - pd.DateOffset(years=lookback_period_years)

    returns_for_ef = monthly_total_returns.loc[start_date_for_ef:end_date_for_ef].dropna(axis=1)

    if returns_for_ef.empty or len(returns_for_ef.columns) < 2:
        print("Not enough data or assets to plot Efficient Frontier for illustration.")
        return

    # Calculate expected returns and sample covariance matrix
    mu = expected_returns.mean_historical_return(returns_for_ef, frequency=12)
    S = risk_models.sample_cov(returns_for_ef, frequency=12)

    # Filter mu and S to ensure they have the same assets in the same order
    common_assets = list(mu.index.intersection(S.columns))
    mu = mu.loc[common_assets]
    S = S.loc[common_assets, common_assets]

    if len(common_assets) < 2:
        print("Not enough common assets after filtering for Efficient Frontier. Skipping.")
        return

    ef = EfficientFrontier(mu, S)

    # Get points on the efficient frontier
    try:
        fig, ax = plt.subplots(figsize=(10, 6))
        plotting.plot_efficient_frontier(ef, ax=ax, show_assets=True)

        # Plot Max Sharpe Ratio portfolio
        current_rf = monthly_rf.loc[end_date_for_ef] if not monthly_rf.empty and end_date_for_ef in monthly_rf.index else 0.0
        ef.max_sharpe(risk_free_rate=current_rf)
        ret_ms, std_ms, _ = ef.portfolio_performance(verbose=False, risk_free_rate=current_rf)
        ax.scatter(std_ms, ret_ms, marker="*", s=200, c="red", label="Max Sharpe")

        # Plot Minimum Volatility portfolio
        ef.min_volatility()
        ret_sv, std_sv, _ = ef.portfolio_performance(verbose=False, risk_free_rate=current_rf)
        ax.scatter(std_sv, ret_sv, marker="*", s=200, c="green", label="Min Volatility")

        ax.set_title(f'Efficient Frontier (Last {lookback_period_years} Years)')
        ax.legend()
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error plotting Efficient Frontier: {e}")

if not monthly_simple_returns_stocks.empty and not monthly_rf_for_backtest.empty:
    plot_efficient_frontier(monthly_simple_returns_stocks, monthly_rf_for_backtest)
else:
    print("Cannot plot Efficient Frontier: Monthly returns or risk-free rate data is empty.")


### 12.5 Portfolio Allocation Heatmap (Illustrative)

Shows the average allocation to each asset over the backtesting period, or a snapshot at a specific rebalancing point, providing insight into which assets were consistently favored by the optimization.

In [ ]:
def plot_portfolio_allocation_heatmap(optimal_weights, num_top_assets=10):
    """
    Plots a heatmap of portfolio allocations over time for the top assets.
    """
    print("Generating portfolio allocation heatmap...")

    if optimal_weights.empty:
        print("No optimal weights to plot heatmap.")
        return

    # Calculate average weights and select top assets
    average_weights = optimal_weights.mean().sort_values(ascending=False)
    top_assets = average_weights.head(num_top_assets).index.tolist()

    if not top_assets:
        print("No top assets to plot in heatmap.")
        return

    # Filter weights for top assets and plot heatmap
    weights_to_plot = optimal_weights[top_assets].transpose()

    # For better visualization, we can use Plotly's heatmap for interactivity
    fig = go.Figure(data=go.Heatmap(
            z=weights_to_plot.values,
            x=weights_to_plot.columns,
            y=weights_to_plot.index,
            colorscale='Viridis'))

    fig.update_layout(
        title_text=f'Portfolio Allocation Heatmap (Top {num_top_assets} Assets)',
        xaxis_title='Date',
        yaxis_title='Asset',
        template='plotly_white',
        height=600
    )
    fig.show()

if not optimal_portfolio_weights.empty:
    plot_portfolio_allocation_heatmap(optimal_portfolio_weights, num_top_assets=len(TICKERS)) # Plot all selected tickers
else:
    print("Cannot plot portfolio allocation heatmap: Optimal weights are empty.")

## 13. Final Deliverables

The successful completion of this project will yield the following deliverables:

1.  **Fully Functional Python Notebook:** A comprehensive Colab notebook (`.ipynb`) containing all the code for data acquisition, processing, factor estimation, portfolio optimization, backtesting, performance attribution, and visualization.
2.  **Processed Data Files:** Cleaned and merged data files (e.g., CSV, HDF5) stored locally, containing stock returns, Fama-French factors, risk-free rates, and monthly factor exposures.
3.  **Performance Report:** A summary of key performance metrics (annualized returns, volatility, Sharpe ratio, Max Drawdown) for both the factor portfolio and the benchmark.
4.  **Attribution Analysis:** Detailed results from CAPM and multi-factor regressions, quantifying alpha and factor betas.
5.  **Interactive Visualizations:** High-quality plots including cumulative returns, rolling performance metrics, factor exposures over time, efficient frontier, and portfolio allocation heatmaps (using `plotly` for interactivity where applicable).
6.  **README.md (Conceptual):** A conceptual README file that would accompany the project on GitHub, detailing installation, usage, project motivation, and results summary.
7.  **Modular Code Structure (Conceptual):** While developed in a notebook, the underlying logic is modular, allowing easy refactoring into Python scripts (`.py` files) for deployment in a production environment.

## 14. Description

**Institutional Factor Investing Engine**

Developed a robust Python-based factor investing engine to construct and optimize equity portfolios using Fama-French factor models. Engineered data pipelines to acquire, clean, and synchronize financial data from Yahoo Finance, Kenneth French Data Library, and FRED. Implemented rolling multi-factor regressions to estimate dynamic factor exposures and employed `PyPortfolioOpt` for monthly portfolio rebalancing, optimizing for maximum Sharpe Ratio under long-only constraints. Conducted comprehensive backtesting, performance attribution (CAPM & multi-factor), and generated interactive visualizations (Plotly) to analyze portfolio performance, risk, and factor tilts. Demonstrated expertise in quantitative finance, statistical modeling, portfolio optimization, and data-driven decision-making in an institutional context.

## 15. Potential Upgrades

This project provides a solid foundation for an institutional factor investing engine. Several enhancements could further improve its robustness, performance, and real-world applicability:

1.  **Expanded Factor Set:** Incorporate additional factors (e.g., Momentum, Quality, Low Volatility) beyond the Fama-French 5-factor model.
2.  **Dynamic Factor Exposure Estimation:** Explore more sophisticated techniques for estimating factor exposures, such as Kalman filters or machine learning models, instead of simple rolling OLS.
3.  **Advanced Portfolio Optimization:**
    *   Implement alternative optimization objectives (e.g., minimum tracking error, risk parity, custom risk/return objectives).
    *   Add more complex constraints (e.g., sector/industry caps, turnover constraints, transaction costs, ESG considerations).
    *   Integrate Black-Litterman model for incorporating investor views.
4.  **Improved Expected Returns and Covariance Estimation:** Utilize advanced methods like Exponentially Weighted Moving Average (EWMA) for covariance, or factor-based models for expected returns and covariance (e.g., Barra-style risk models).
5.  **Alternative Backtesting Methodologies:** Implement full rebalancing backtest (asset-level) and incorporate more realistic trading assumptions (slippage, liquidity).
6.  **Out-of-Sample Validation:** Strictly separate data into in-sample (for model training/parameter tuning) and out-of-sample (for final performance evaluation) periods.
7.  **Robustness Checks:** Perform sensitivity analysis on key parameters (e.g., rolling window size, lookback periods, optimization constraints).
8.  **Data Quality & Error Handling:** Enhance error handling for API calls, data inconsistencies, and missing data imputation.
9.  **Deployment & Monitoring:** Containerize the application (Docker) and deploy it to a cloud platform (GCP, AWS, Azure) for automated execution and performance monitoring.
10. **Interactive Dashboard:** Develop a dedicated interactive dashboard using Streamlit, Dash, or Plotly Dash for real-time portfolio monitoring and drill-down analysis.
11. **Fundamental Data Integration:** Incorporate fundamental financial data (e.g., balance sheet, income statement) to build proprietary factors or improve stock selection.
12. **Machine Learning Integration:** Use ML models for alpha generation, risk prediction, or factor timing.

## 13. Final Deliverables

The successful completion of this project will yield the following deliverables:

1.  **Fully Functional Python Notebook:** A comprehensive Colab notebook (`.ipynb`) containing all the code for data acquisition, processing, factor estimation, portfolio optimization, backtesting, performance attribution, and visualization.
2.  **Processed Data Files:** Cleaned and merged data files (e.g., CSV, HDF5) stored locally, containing stock returns, Fama-French factors, risk-free rates, and monthly factor exposures.
3.  **Performance Report:** A summary of key performance metrics (annualized returns, volatility, Sharpe ratio, Max Drawdown) for both the factor portfolio and the benchmark.
4.  **Attribution Analysis:** Detailed results from CAPM and multi-factor regressions, quantifying alpha and factor betas.
5.  **Interactive Visualizations:** High-quality plots including cumulative returns, rolling performance metrics, factor exposures over time, efficient frontier, and portfolio allocation heatmaps (using `plotly` for interactivity where applicable).
6.  **README.md (Conceptual):** A conceptual README file that would accompany the project on GitHub, detailing installation, usage, project motivation, and results summary.
7.  **Modular Code Structure (Conceptual):** While developed in a notebook, the underlying logic is modular, allowing easy refactoring into Python scripts (`.py` files) for deployment in a production environment.

## 14. Description

**Institutional Factor Investing Engine**

Developed a robust Python-based factor investing engine to construct and optimize equity portfolios using Fama-French factor models. Engineered data pipelines to acquire, clean, and synchronize financial data from Yahoo Finance, Kenneth French Data Library, and FRED. Implemented rolling multi-factor regressions to estimate dynamic factor exposures and employed `PyPortfolioOpt` for monthly portfolio rebalancing, optimizing for maximum Sharpe Ratio under long-only constraints. Conducted comprehensive backtesting, performance attribution (CAPM & multi-factor), and generated interactive visualizations (Plotly) to analyze portfolio performance, risk, and factor tilts. Demonstrated expertise in quantitative finance, statistical modeling, portfolio optimization, and data-driven decision-making in an institutional context.

## 15. Potential Upgrades

This project provides a solid foundation for an institutional factor investing engine. Several enhancements could further improve its robustness, performance, and real-world applicability:

1.  **Expanded Factor Set:** Incorporate additional factors (e.g., Momentum, Quality, Low Volatility) beyond the Fama-French 5-factor model.
2.  **Dynamic Factor Exposure Estimation:** Explore more sophisticated techniques for estimating factor exposures, such as Kalman filters or machine learning models, instead of simple rolling OLS.
3.  **Advanced Portfolio Optimization:**
    *   Implement alternative optimization objectives (e.g., minimum tracking error, risk parity, custom risk/return objectives).
    *   Add more complex constraints (e.g., sector/industry caps, turnover constraints, transaction costs, ESG considerations).
    *   Integrate Black-Litterman model for incorporating investor views.
4.  **Improved Expected Returns and Covariance Estimation:** Utilize advanced methods like Exponentially Weighted Moving Average (EWMA) for covariance, or factor-based models for expected returns and covariance (e.g., Barra-style risk models).
5.  **Alternative Backtesting Methodologies:** Implement full rebalancing backtest (asset-level) and incorporate more realistic trading assumptions (slippage, liquidity).
6.  **Out-of-Sample Validation:** Strictly separate data into in-sample (for model training/parameter tuning) and out-of-sample (for final performance evaluation) periods.
7.  **Robustness Checks:** Perform sensitivity analysis on key parameters (e.g., rolling window size, lookback periods, optimization constraints).
8.  **Data Quality & Error Handling:** Enhance error handling for API calls, data inconsistencies, and missing data imputation.
9.  **Deployment & Monitoring:** Containerize the application (Docker) and deploy it to a cloud platform (GCP, AWS, Azure) for automated execution and performance monitoring.
10. **Interactive Dashboard:** Develop a dedicated interactive dashboard using Streamlit, Dash, or Plotly Dash for real-time portfolio monitoring and drill-down analysis.
11. **Fundamental Data Integration:** Incorporate fundamental financial data (e.g., balance sheet, income statement) to build proprietary factors or improve stock selection.
12. **Machine Learning Integration:** Use ML models for alpha generation, risk prediction, or factor timing.